# All-sky animation of the UVEX transients

## Overview

This notebook turns a simulation run into a movie of the whole survey. It borrows its sky panel from the
[UVEX scheduler's `animate.ipynb`](https://github.com/m4opt/uvex-scheduler/blob/main/notebooks/animate.ipynb), but
instead of tracking the completion of the survey it puts the **transients** on the sky and counts what UVEX finds.

- **Sky panel:** an all-sky Aitoff map of the UVEX fields. It builds up observation by observation, shaded by the number
  of visits each patch of sky has received. Every simulated transient that UVEX eventually detects is a **dot**,
  coloured by type, that appears at its explosion, brightens to a peak and fades. A dot gets a white rim at the
  moment of its first detection, together with a short expanding ring.
- **Bars:** one vertical bar per transient type that **fills** as events of that type are detected. Each bar shows the
  running *expected number of real events* (see the caveats below).

It produces two outputs:

1. A **static figure** of the final frame: the whole survey's coverage, every dot at its brightest, and the full bars.
2. An **animation** (`mp4`, or `gif` if `ffmpeg` is missing) of the survey from its first to its last observation.

### Required data

| Item | Where it comes from | Default |
| --- | --- | --- |
| An **event summary** table | the `event_summary.ecsv` of a pipeline run (`uvex-transients run ...`), or a release download | the summary of release `v0.6.5alpha` |
| The `SurveySchedule` the run was made with | `uvex_transients.surveys.get_schedule` | the package's default schedule |

By default the notebook downloads the `event_summary.ecsv` of release `v0.6.5alpha` (8 MB) with
`uvex_transients.utils.get_results`. Set `RELEASE_TAG` to use another release (the newest, `RELEASE_TAG = None`, may not
carry the summary yet), or `SUMMARY_PATH` to a file path to use your own. The summary holds only the events that
cleared the detection screens, so **every dot is an event that is detected sooner or later**. Nothing that UVEX misses
is drawn.

### How a dot's lifetime and brightness are set

The simulation stores *when each event was observed*, not a full light curve, so a dot's life is schematic and is built
from three times in the summary table:

- it **starts** at the explosion time `t_explosion`,
- it **peaks** at `peak_time`, the epoch of the event's best detection, and
- it **ends** at the *later* of the last detection `t_last_det` and `t_explosion + MIN_DURATION[type]`.

The brightness rises linearly from the start to the peak and then fades to the end (see `DECAY_POWER`). The minimum
duration is set per type at the top of the notebook. It is what keeps fast events, which are often seen for only a day
or two, from flickering by in a single frame, while slow ones (TDEs, superluminous supernovae) still last as long as
they were really detected. Dot brightness is *not* comparable between events: every dot reaches the same peak
brightness.

### Caveats

- **Dots are simulated events, bars are expected yield.** Each row of the summary is one simulated event, and rare
  classes are sampled more densely than they occur. A row's `weight` is the expected number of real events it stands
  for, and the bars add up those weights, so the bar heights are the report's yields (the last cell of the data
  processing section checks this against `estimate_yield`). The number of dots of a type is therefore not the number
  on its bar.
- The schedule must be the one the run was made with, or the visit map will not match the detections.
- Writing an `mp4` needs `ffmpeg` on your path. If it is missing the notebook falls back to a `gif`, which is slower to
  write and larger.
- The style cell uses LaTeX for text when a TeX installation is found, and matplotlib's own Computer Modern otherwise.
- The animation holds one frame per `1 / FPS` second of movie, and the whole survey (about 2 years) is spread over
  `DURATION_S` seconds. At the defaults one frame is about 0.8 days.

## Setup and Configuration

In [ ]:
import warnings
from pathlib import Path

import astropy.units as u
import erfa
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import ICRS, SkyCoord
from astropy.table import QTable
from astropy.time import Time
from astropy_healpix import HEALPix
from IPython import display as ipython_display
from IPython.display import Image, Video
from ligo.skymap import plot  # noqa: F401  (registers the "astro aitoff" projection)
from m4opt.fov import footprint_healpix
from matplotlib import animation
from matplotlib.colors import LogNorm, to_rgba_array
from tqdm.auto import tqdm

from uvex_transients.simulation.rates import estimate_yield
from uvex_transients.surveys import get_schedule
from uvex_transients.utils import get_results

# The survey runs to the year 2030 and beyond, which ERFA flags as a "dubious year" on every time conversion.
warnings.filterwarnings("ignore", category=erfa.ErfaWarning)

In [ ]:
# ---------------------------------------------------------------------------------------------- #
# UVEX Transients notebook style. Copy this cell unchanged into every notebook (see the readme), #
# so that all of them look the same.                                                             #
# ---------------------------------------------------------------------------------------------- #
import shutil

import matplotlib as mpl
from cycler import cycler
from matplotlib.colors import LinearSegmentedColormap

# The UVEX logo gradient runs from blue (top) to violet (bottom). These are its colors, sampled from the logo files.
UVEX_BLUE = "#72AEFF"
UVEX_VIOLET = "#AF6EFF"
# The same hues, deepened so that thin lines and small markers stay legible on a white background.
UVEX_BLUE_DEEP = "#3B87EB"
UVEX_VIOLET_DEEP = "#9742ED"
UVEX_NAVY = "#0A0B1E"  # A dark background for images and maps.

# Default color cycle: the two logo hues first, then accents that stay apart from them and from each other.
UVEX_CYCLE = [UVEX_BLUE_DEEP, UVEX_VIOLET_DEEP, "#13A89E", "#F2A93B", "#E5509A", "#475569"]

# Sequential colormap for images and maps, "uvex": indigo (low) through violet and blue to pale (high). Its lightness
# rises steadily, so it also reads in grayscale. The low end is lighter than UVEX_NAVY, so faint pixels stay visible.
UVEX_CMAP = LinearSegmentedColormap.from_list("uvex", ["#2A2078", "#4B2BA8", UVEX_VIOLET, UVEX_BLUE, "#EAF4FF"])
mpl.colormaps.register(UVEX_CMAP, name="uvex", force=True)

# Text is set with LaTeX when a TeX installation is found, and with matplotlib's own Computer Modern look-alike if not.
USE_LATEX = shutil.which("latex") is not None and shutil.which("dvipng") is not None

mpl.rcParams.update(
    {
        # --- Publication style (AAS two-column-ish, LaTeX/Computer Modern) ---
        "text.usetex": USE_LATEX,
        "text.latex.preamble": r"\usepackage{amsmath,amssymb}",
        "font.family": "serif",
        "font.serif": ["Computer Modern Roman", "cmr10", "DejaVu Serif"],
        "mathtext.fontset": "cm",
        "axes.formatter.use_mathtext": True,
        "axes.unicode_minus": False,
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 8,
        "axes.linewidth": 0.6,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.top": True,
        "ytick.right": True,
        "xtick.major.size": 3,
        "xtick.minor.size": 1.5,
        "ytick.major.size": 3,
        "ytick.minor.size": 1.5,
        "lines.linewidth": 1.0,
        "figure.dpi": 150,
        "savefig.dpi": 300,
        # --- UVEX colors ---
        "axes.prop_cycle": cycler(color=UVEX_CYCLE),
        "image.cmap": "uvex",
        "legend.frameon": False,
    }
)


def tex(text):
    """Escape the characters LaTeX treats specially (_ % & #), so plain text can go into labels when LaTeX is on."""
    if mpl.rcParams["text.usetex"]:
        for char in "_%&#":
            text = text.replace(char, "\\" + char)
    return text

In [ ]:
# ------------------------------------------------------------------ #
# User settings. Everything you are likely to want to change is here. #
# ------------------------------------------------------------------ #

# --- Data ---
# SUMMARY_PATH: "release" downloads a GitHub release's event summary, whose rows are the events that were detected.
#               A file path loads your own event_summary.ecsv.
SUMMARY_PATH = "release"
RELEASE_TAG = "v0.6.5alpha"  # The release to download. None: the newest, which may not carry the summary yet.
SCHEDULE_NAME = None  # Name of the schedule the run was made with. None: the package default.

# --- Transient types ---
# Display name and color of every type, in the order the bars are drawn (left to right). The color families follow the
# classes: blues and teals for type I supernovae, greens and yellow for type II, violet for superluminous supernovae,
# warm colors for the exotic transients. A type found in the summary but missing here gets a gray and its own key as
# its name.
TRANSIENT_STYLE = {
    "ia_sne": ("Type Ia", "#4DA3FF"),
    "ib_sne": ("Type Ib", "#7EE0F5"),
    "ic_sne": ("Type Ic", "#2EC4B6"),
    "icbl_sne": ("Type Ic-BL", "#A5F3D0"),
    "iip_sne": ("Type IIP", "#A8E05F"),
    "iib_sne": ("Type IIb", "#E4E85A"),
    "slsne": ("SLSN-I", "#B07CFF"),
    "tde": ("TDE", "#FFB03B"),
    "llgrb": ("llGRB", "#FF5A5F"),
    "lfbot": ("LFBOT", "#FF7AC6"),
    "kilonova": ("Kilonova", "#D9D9D9"),
}

# --- Dot lifetimes ---
# Each dot lasts until the later of the event's last detection and `t_explosion + MIN_DURATION[type]`, in days. Raise a
# value to make that type linger on screen, lower it to let it flash by. Types missing here get the default.
MIN_DURATION = {
    "ia_sne": 40.0,
    "ib_sne": 30.0,
    "ic_sne": 30.0,
    "icbl_sne": 30.0,
    "iip_sne": 30.0,
    "iib_sne": 30.0,
    "slsne": 120.0,
    "tde": 100.0,
    "llgrb": 10.0,
    "lfbot": 15.0,
    "kilonova": 7.0,
}
DEFAULT_MIN_DURATION = 20.0
DECAY_POWER = 2.0  # Brightness after the peak falls as (1 - x)^DECAY_POWER, x the fraction of the fade. 1: linear.
TAIL_DAYS = 30.0  # The movie runs this many days past the last observation, so the last dots can fade out.

# --- Dots ---
DOT_SIZE = (1.5, 26.0)  # Marker area in points^2 of a dot that is just starting, and of one at its peak.
DOT_ALPHA = (0.15, 1.0)  # Opacity of the same two. Together with the size, this is the brightness of a dot.
SHOW_PULSE = True  # Draw a ring that expands from a dot at the moment of its first detection.
PULSE_DAYS = 4.0  # Days the ring takes to expand and fade.
PULSE_GROWTH = 8.0  # Area of the ring at the end of the pulse, relative to a dot at its peak.

# --- Sky panel ---
# HPX_NSIDE: HEALPix resolution of the visit map. 1024 (0.06 deg per pixel) is what the scheduler's animation uses, and
#            shows the gaps between detector chips. Building the map takes a few minutes, and the time grows with
#            the square of NSIDE: 256 takes about a fifth as long and is already finer than the image.
HPX_NSIDE = 1024
CENTER_RA_HOURS = 8.0  # Right ascension at the centre of the Aitoff map, in hours.
MAP_CMAP = "uvex"  # A matplotlib colormap name for the visit map. "uvex" is the standard map from the style cell.
MAP_VMAX = None  # Number of visits at the top of the (logarithmic) color scale. None: the most visited pixel.

# --- Bars ---
# BAR_SCALE: "own"    every bar fills to the top at its type's final yield, so the bars are equally tall and the number
#                     above each one says how much it holds. The best choice when yields span orders of magnitude.
#            "log"    one shared logarithmic axis, so the heights compare the types.
#            "linear" one shared linear axis. Rare types stay near zero.
BAR_SCALE = "own"

# --- Output ---
OUTPUT_DIR = Path("../../results")  # Repo-level results/ (gitignored). Where the figure and the animation are written.
SAVE_STATIC = True  # Write the static figure of the final frame.
MAKE_ANIMATION = True  # Set False to skip the (slow) animation.
ANIMATION_FORMAT = "mp4"  # "mp4" (needs ffmpeg) or "gif".
DURATION_S = 30  # Length of the movie in seconds.
FPS = 30  # Frames per second.
PIXEL_HEIGHT = 1080  # Height of the movie in pixels. The width follows from ASPECT.
ASPECT = 16 / 9  # Width over height of the movie.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Figure geometry. The size in inches is fixed, so the text and the layout look the same at every resolution, and
# PIXEL_HEIGHT only sets the dots per inch. The pixel size is rounded to even numbers, which every video codec accepts.
FIG_HEIGHT = 7.2  # inches
pixel_height = 2 * round(PIXEL_HEIGHT / 2)
pixel_width = 2 * round(pixel_height * ASPECT / 2)
FIG_SIZE = (FIG_HEIGHT * pixel_width / pixel_height, FIG_HEIGHT)
FIG_DPI = pixel_height / FIG_HEIGHT

# Where the sky axes sit in the figure, as fractions of it. Their size sets the pixel grid of the all-sky projection,
# so the lookup table of the visit map must be made on axes of exactly this size. `add_sky_axes` is the one place that
# makes them.
SKY_RECT = {"left": 0.02, "right": 0.735, "top": 0.9, "bottom": 0.12}


def add_sky_axes(fig):
    """Add the all-sky Aitoff axes to `fig`, in the same place and with the same projection every time."""
    sky_grid = fig.add_gridspec(1, 1, **SKY_RECT)
    center = SkyCoord(CENTER_RA_HOURS * u.hourangle, 0 * u.deg)
    return fig.add_subplot(sky_grid[0], projection="astro aitoff", center=center)


# A dark look for the movie. The sizes are larger than the style cell's because this figure is far bigger than a
# journal column. The settings are applied with `rc_context`, so that they do not leak into other cells.
DARK = {
    "figure.facecolor": UVEX_NAVY,
    "savefig.facecolor": UVEX_NAVY,
    "axes.facecolor": UVEX_NAVY,
    "axes.edgecolor": "0.75",
    "axes.labelcolor": "0.92",
    "text.color": "0.92",
    "xtick.color": "0.8",
    "ytick.color": "0.8",
    "grid.color": "0.6",
    "font.size": 14,
    "axes.labelsize": 14,
    "axes.titlesize": 15,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
}

## Data Loading

The event summary and the survey schedule are loaded here. Only the summary's columns are read: the schedule is used
for the footprint of every observation.

In [ ]:
schedule = get_schedule(SCHEDULE_NAME)

summary_path = get_results(["summary"], tag=RELEASE_TAG)["summary"] if SUMMARY_PATH == "release" else SUMMARY_PATH
summary = QTable.read(summary_path)

print(f"Summary: {len(summary)} detected events from {summary_path}")
print(f"Schedule: {schedule}")

## Data Processing

Four things are prepared:

1. **The events.** The positions, types and the three times that set each dot's life.
2. **The frames.** An even grid of times from the start of the survey to a little after its end.
3. **The bars.** The cumulative expected yield of each type at every frame.
4. **The visit map.** Which HEALPix pixels each observation covered, grouped by the frame it falls in, so that a frame
   only adds its own observations to a running map. A map at `NSIDE = 1024` has 12.6 million pixels and the
   footprints add up to some 200 million hits, far more than the picture can show (it is about 1900 pixels wide).
   So the footprints are rasterized in chunks and only the hits on the pixels the image actually displays are kept,
   which is about a tenth of them and is exactly what the picture would show.

In [ ]:
# --- 1. The events ---
types = np.asarray(summary["transient_type"]).astype(str)

# Types in the display order, then any the summary has that the settings do not know.
style = dict(TRANSIENT_STYLE)
for name in sorted(set(types) - set(style)):
    style[name] = (name, "#9AA0A6")
style = {name: value for name, value in style.items() if name in set(types)}
type_names = list(style)
type_labels = [style[name][0] for name in type_names]
type_colors = [style[name][1] for name in type_names]
type_index = np.array([type_names.index(name) for name in types])

ra = np.asarray(summary["coord"].ra.deg)
dec = np.asarray(summary["coord"].dec.deg)
weight = np.asarray(summary["weight"], dtype=float)

t_explosion = np.asarray(summary["t_explosion"].jd)
t_first_det = np.asarray(summary["t_first_det"].jd)
t_last_det = np.asarray(summary["t_last_det"].jd)
t_peak_det = np.asarray(summary["peak_time"].jd)

# A dot starts at the explosion and ends at the later of the last detection and the type's minimum duration. Its peak
# is the best detection, kept inside that window.
min_duration = np.array([MIN_DURATION.get(name, DEFAULT_MIN_DURATION) for name in types])
t_start = t_explosion
t_end = np.maximum(t_last_det, t_explosion + min_duration)
t_peak = np.clip(t_peak_det, t_start, t_end)

# Draw the common types first, so that the rare ones end up on top of them.
n_per_type = np.bincount(type_index, minlength=len(type_names))
draw_order = np.argsort(-n_per_type[type_index], kind="stable")
(ra, dec, weight, type_index, t_explosion, t_first_det, t_start, t_end, t_peak) = (
    array[draw_order] for array in (ra, dec, weight, type_index, t_explosion, t_first_det, t_start, t_end, t_peak)
)
base_rgba = to_rgba_array([type_colors[i] for i in type_index])
print(f"{len(types)} events of {len(type_names)} types; median dot lifetime {np.median(t_end - t_start):.0f} d.")

In [ ]:
def intensity(jd, rows):
    """Return the brightness, 0 to 1, of the events `rows` at time `jd`: a linear rise to the peak, then a fade."""
    rise = np.clip((jd - t_start[rows]) / np.maximum(t_peak[rows] - t_start[rows], 1e-6), 0, 1)
    fade = np.clip(1 - (jd - t_peak[rows]) / np.maximum(t_end[rows] - t_peak[rows], 1e-6), 0, 1) ** DECAY_POWER
    return np.where(jd < t_peak[rows], rise, fade)


# --- 2. The frames: the survey, then a tail for the last dots to fade ---
n_frames = round(DURATION_S * FPS) + 1
frame_jd = np.linspace(schedule.start_time.jd, schedule.end_time.jd + TAIL_DAYS, n_frames)
frame_date = Time(frame_jd, format="jd").strftime("%Y-%m-%d")
print(f"{n_frames} frames, {np.diff(frame_jd)[0]:.2f} days per frame, {frame_date[0]} to {frame_date[-1]}.")

# --- 3. The bars: the expected yield of each type detected by every frame ---
bar_values = np.zeros((n_frames, len(type_names)))
for j in range(len(type_names)):
    of_type = type_index == j
    order = np.argsort(t_first_det[of_type])
    cumulative = np.concatenate([[0.0], np.cumsum(weight[of_type][order])])
    bar_values[:, j] = cumulative[np.searchsorted(t_first_det[of_type][order], frame_jd, side="right")]

# The last frame comes after every first detection, so it should be the report's yield of each type.
check = estimate_yield(summary)
yield_by_type = dict(zip(check["transient_type"], check["expected_events"]))
final = QTable(
    {
        "type": type_names,
        "dots": n_per_type,
        "bar": np.round(bar_values[-1], 1),
        "estimate_yield": np.round([yield_by_type[name] for name in type_names], 1),
    }
)
final

In [ ]:
# --- 4. The visit map ---
nside = HPX_NSIDE
n_pix = 12 * nside**2

# Which HEALPix pixel does every pixel of the image show? Draw a map of pixel indices onto a throwaway figure with the
# nearest neighbour, which reads one pixel without mixing two. Pixel numbers are exact in a float.
probe_fig = plt.figure(figsize=FIG_SIZE, dpi=FIG_DPI)
probe_ax = add_sky_axes(probe_fig)
probe = probe_ax.imshow_hpx(np.arange(n_pix, dtype=float), order="nearest-neighbor", nested=False)
probe_array = probe.get_array()
raster_shape = probe_array.shape
probe_wcs = probe_ax.wcs
on_sky = ~np.ma.getmaskarray(probe_array)
shown = np.ma.getdata(probe_array)[on_sky].astype(np.intp)
plt.close(probe_fig)

# Only these pixels are ever displayed, so only they need counting. `compact` renumbers them 0, 1, 2, ...
displayed = np.unique(shown)
compact = np.full(n_pix, -1, dtype=np.int32)
compact[displayed] = np.arange(len(displayed), dtype=np.int32)
image_pixel = np.zeros(raster_shape, dtype=np.intp)  # The compact pixel behind each image pixel.
image_pixel[on_sky] = compact[shown]
print(f"The image shows {len(displayed)} of the {n_pix} HEALPix pixels.")

# Check the lookup against the axes' own WCS: the sky position of an image pixel must lie inside the HEALPix pixel that
# the table says is shown there.
rng = np.random.default_rng(0)
rows, cols = np.nonzero(on_sky)
sample = rng.choice(len(rows), size=2000, replace=False)
position = probe_wcs.pixel_to_world(cols[sample], rows[sample]).icrs
expected = HEALPix(nside=nside, order="ring", frame=ICRS()).skycoord_to_healpix(position)
agreement = np.mean(expected == np.ma.getdata(probe_array)[rows[sample], cols[sample]].astype(np.intp))
print(f"{agreement:.1%} of 2000 sampled image pixels show the HEALPix pixel at their sky position.")
if agreement < 0.995:  # A pixel centre exactly on a pixel boundary can round either way, so allow a few.
    raise RuntimeError("The lookup table does not match the sky positions of the image pixels.")

# The footprint of every observation, in chunks. The frame an observation belongs to is the first one at or after its
# start. Keep a hit only if it lands on a displayed pixel.
observations = schedule.observe_rows
observe_jd = observations["start_time"].jd
frame_of_observation = np.searchsorted(frame_jd, observe_jd, side="left")
healpix = HEALPix(nside=nside, order="ring", frame=ICRS())
CHUNK = 250  # Observations per call, which keeps the lists of pixel numbers small.

kept_pixel, kept_frame, n_hits = [], [], 0
for first in tqdm(range(0, len(observations), CHUNK), desc="Footprints"):
    chunk = slice(first, first + CHUNK)
    pixel_lists = footprint_healpix(
        healpix, schedule.fov, observations["target_coord"][chunk], observations["roll"][chunk]
    )
    pixels = np.concatenate(pixel_lists)
    frames = np.repeat(frame_of_observation[chunk], [len(pixel_list) for pixel_list in pixel_lists])
    visible = compact[pixels] >= 0
    kept_pixel.append(compact[pixels][visible])
    kept_frame.append(frames[visible].astype(np.int32))
    n_hits += len(pixels)

hit_pixel = np.concatenate(kept_pixel)
hit_frame = np.concatenate(kept_frame)
del kept_pixel, kept_frame
by_frame = np.argsort(hit_frame, kind="stable")
hit_pixel = hit_pixel[by_frame]
frame_edges = np.concatenate([[0], np.cumsum(np.bincount(hit_frame, minlength=n_frames + 1))])  # Frame k: [k], [k+1].
max_visits = int(np.bincount(hit_pixel, minlength=len(displayed)).max())
print(f"{len(observations)} observations, {n_hits} pixel hits, {len(hit_pixel)} of them on displayed pixels.")
print(f"The most visited pixel has {max_visits} visits.")


class VisitMap:
    """The number of visits to every displayed pixel at a frame, built up one frame at a time."""

    def __init__(self):
        self.counts = np.zeros(len(displayed), dtype=np.int32)
        self.frame = -1

    def at(self, frame):
        if frame == self.frame + 1:  # The usual case of an animation: add the hits of one frame.
            hits = hit_pixel[frame_edges[frame] : frame_edges[frame + 1]]
            self.counts += np.bincount(hits, minlength=len(displayed)).astype(np.int32)
        elif frame != self.frame:  # A jump, such as the final frame of the static figure: start over.
            self.counts = np.bincount(hit_pixel[: frame_edges[frame + 1]], minlength=len(displayed)).astype(np.int32)
        self.frame = frame
        return self.counts

## Visualization

One function builds the figure and another, `update`, sets it to any frame, so the static figure is just `update` at
the last frame and the animation is `update` at every frame.

Two choices keep the animation fast:

- The visit map is an ordinary image that is created **once**. The data processing section already worked out which
  HEALPix pixel every image pixel shows, so a frame only gathers the current counts through that table and hands them to
  `set_array`. Calling `imshow_hpx` for every frame would re-project the whole map each time.
- The dots are **one** scatter collection whose positions, sizes and colors are replaced at every frame, and only the
  dots that are alive at that time are in it.

In [ ]:
def make_figure():
    """Create the figure and return it with its artists and a function that sets it to a frame."""
    with plt.rc_context(DARK):
        fig = plt.figure(figsize=FIG_SIZE, dpi=FIG_DPI)
        # The bars have their own grid, lower than the sky's, to leave room for the rotated type names under them.
        bar_grid = fig.add_gridspec(1, 1, left=0.79, right=0.985, top=0.9, bottom=0.2)

        # --- The sky ---
        ax_sky = add_sky_axes(fig)
        ax_sky.patch.set_facecolor("#10122E")
        ax_sky.grid(color="0.6", alpha=0.25, linewidth=0.5)

        # The visit map starts empty (everything masked, so the background shows). Its shape is the one of the lookup
        # table from the data processing, which was drawn on an axes of the same projection.
        vmax = MAP_VMAX if MAP_VMAX is not None else max_visits
        image = ax_sky.imshow(
            np.ma.masked_all(raster_shape), norm=LogNorm(vmin=1, vmax=max(vmax, 2), clip=True), cmap=MAP_CMAP
        )
        # The image must sit on the pixel grid of these axes, or it would be cropped or stretched against the dots.
        if (ax_sky.header["NAXIS2"], ax_sky.header["NAXIS1"]) != raster_shape:
            raise RuntimeError("The sky axes do not have the pixel grid the lookup table was made for.")
        colorbar = fig.colorbar(image, ax=ax_sky, orientation="horizontal", fraction=0.04, pad=0.03, aspect=50)
        colorbar.set_label("Number of visits")

        to_sky = ax_sky.get_transform("world")
        dots = ax_sky.scatter([], [], s=[], transform=to_sky, linewidths=0.6, zorder=3)
        rings = ax_sky.scatter([], [], s=[], transform=to_sky, facecolors="none", linewidths=1.0, zorder=4)
        clock = ax_sky.text(0.02, 1.0, "", transform=ax_sky.transAxes, va="bottom", fontsize=16, usetex=False)

        # --- The bars ---
        ax_bar = fig.add_subplot(bar_grid[0])
        x = np.arange(len(type_names))
        final_values = bar_values[-1]
        if BAR_SCALE == "own":
            top, floor, y_label = 1.0, 0.0, "Fraction of the final yield"
            to_height = lambda value: value / final_values  # noqa: E731
        elif BAR_SCALE == "log":
            top, floor, y_label = final_values.max(), 0.5, "Expected detections"
            to_height = lambda value: np.maximum(value, floor)  # noqa: E731
        elif BAR_SCALE == "linear":
            top, floor, y_label = final_values.max(), 0.0, "Expected detections"
            to_height = lambda value: value  # noqa: E731
        else:
            raise ValueError(f"BAR_SCALE must be 'own', 'log' or 'linear', not {BAR_SCALE!r}.")

        bars = ax_bar.bar(x, np.zeros(len(x)) + floor, bottom=0, width=0.78, color=type_colors, linewidth=0)
        if BAR_SCALE == "log":
            ax_bar.set_yscale("log")
            ax_bar.set_ylim(floor, top * 12)
        else:
            ax_bar.set_ylim(0, top * 1.28)
            ax_bar.set_yticks([0, 0.5 * top, top] if BAR_SCALE == "own" else ax_bar.get_yticks())
        ax_bar.set_xlim(-0.6, len(x) - 0.4)
        ax_bar.set_xticks(x)
        ax_bar.set_xticklabels([tex(label) for label in type_labels], rotation=90, fontsize=11)
        ax_bar.set_ylabel(y_label)
        ax_bar.set_title("Expected UVEX detections", fontsize=14)
        ax_bar.tick_params(axis="x", length=0)
        for side in ("top", "right"):
            ax_bar.spines[side].set_visible(False)
        ax_bar.tick_params(which="both", top=False, right=False)
        counts = [
            ax_bar.text(xi, floor, "", rotation=90, ha="center", va="bottom", fontsize=12, usetex=False) for xi in x
        ]

    visits = VisitMap()

    def update(frame, pulses=SHOW_PULSE):
        """Set the artists to the state of the survey at one frame, with or without the detection rings."""
        jd = frame_jd[frame]

        # The visit map, through the lookup table. Pixels no one has visited are masked, so the background shows.
        values = np.where(on_sky, visits.at(frame)[image_pixel], 0)
        image.set_array(np.ma.masked_less(values, 1))

        # The dots that are alive now, brightened by their light curve, with a rim once they have been detected.
        alive = np.flatnonzero((t_start <= jd) & (jd <= t_end))
        bright = intensity(jd, alive)
        alpha = DOT_ALPHA[0] + (DOT_ALPHA[1] - DOT_ALPHA[0]) * bright
        face = base_rgba[alive].copy()
        face[:, 3] = alpha
        rim = np.ones_like(face)
        rim[:, 3] = np.where(t_first_det[alive] <= jd, alpha, 0.0)
        dots.set_offsets(np.column_stack([ra[alive], dec[alive]]))
        dots.set_sizes(DOT_SIZE[0] + (DOT_SIZE[1] - DOT_SIZE[0]) * np.sqrt(bright))
        dots.set_facecolors(face)
        dots.set_edgecolors(rim)

        # The rings: a short pulse from every dot at its first detection.
        since = jd - t_first_det
        pulsing = np.flatnonzero((since >= 0) & (since < PULSE_DAYS)) if pulses else np.array([], dtype=int)
        progress = since[pulsing] / PULSE_DAYS
        ring_color = base_rgba[pulsing].copy()
        ring_color[:, 3] = 1 - progress
        rings.set_offsets(np.column_stack([ra[pulsing], dec[pulsing]]))
        rings.set_sizes(DOT_SIZE[1] * (1 + (PULSE_GROWTH - 1) * progress))
        rings.set_edgecolors(ring_color)

        # The bars and the number above each.
        heights = to_height(bar_values[frame])
        label_y = heights * 1.15 if BAR_SCALE == "log" else heights + 0.02 * top
        for j, (bar, count) in enumerate(zip(bars, counts)):
            bar.set_height(heights[j])
            count.set_position((x[j], label_y[j]))
            value = bar_values[frame, j]
            count.set_text(f"{value:,.0f}" if value >= 10 else f"{value:.1f}")

        clock.set_text(f"{frame_date[frame]}   (day {jd - frame_jd[0]:.0f})")
        return image, dots, rings, clock, *bars, *counts

    return fig, update

### Static figure

The survey at its end: the visit map after every observation, each bar at its final yield, and the dots that are still
alive then, which are the events of the last few months (the slow ones, TDEs and superluminous supernovae, reach back
furthest). The detection rings are left out, since a still frame has no motion for them to show.

In [ ]:
# The first frame at or after the end of the survey: every observation and detection is in, and the dots of the last
# months are still on the sky, since the tail has not run yet.
survey_end_frame = int(np.searchsorted(frame_jd, schedule.end_time.jd, side="left"))

fig, update = make_figure()
update(survey_end_frame, pulses=False)
if SAVE_STATIC:
    fig.savefig(OUTPUT_DIR / "all_sky_transients_static.png", dpi=FIG_DPI)
plt.show()

### Animation

The movie runs from the first observation to `TAIL_DAYS` after the last, one frame per `1 / FPS` seconds. Writing it is the
slow step: every frame is a full-resolution figure, so expect a few minutes at the defaults.

In [ ]:
if MAKE_ANIMATION:
    fig, update = make_figure()

    use_mp4 = ANIMATION_FORMAT == "mp4" and animation.writers.is_available("ffmpeg")
    if ANIMATION_FORMAT == "mp4" and not use_mp4:
        print("ffmpeg was not found; writing a gif instead.")
    suffix = "mp4" if use_mp4 else "gif"
    movie_path = OUTPUT_DIR / f"all_sky_transients.{suffix}"
    # yuv420p is the pixel format that every player, QuickTime included, can show.
    writer = (
        animation.FFMpegWriter(fps=FPS, codec="libx264", extra_args=["-pix_fmt", "yuv420p"])
        if use_mp4
        else animation.PillowWriter(fps=FPS)
    )

    with tqdm(total=n_frames, leave=False) as progress:

        def advance(frame, _total):
            """Move the progress bar to the frame that was just written."""
            progress.n = frame + 1
            progress.refresh()

        anim = animation.FuncAnimation(fig, update, frames=n_frames, blit=False)
        anim.save(movie_path, writer=writer, dpi=FIG_DPI, progress_callback=advance)
    plt.close(fig)

    print(f"Wrote {movie_path} ({n_frames} frames, {n_frames / FPS:.0f} s).")
    ipython_display.display(
        Video(str(movie_path), embed=True, html_attributes="controls loop")
        if use_mp4
        else Image(filename=str(movie_path))
    )